# 02 - Data Preprocessing
## Amazon Product Intelligence System

**Objective:**  
Transform the raw, messy review data into a clean, structured dataset ready for all three NLP components: topic modeling, semantic retrieval, and classification.

All data quality issues identified in `01_eda.ipynb` are addressed here systematically.

**Input:** `data/raw/Reviews.csv`  
**Output:** `data/processed/reviews_clean.csv`

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

print('All libraries imported successfully.')

All libraries imported successfully.


In [2]:
# Load raw data
df = pd.read_csv('../data/raw/Reviews.csv')

print(f'Raw dataset shape: {df.shape}')
print(f'Total reviews: {len(df):,}')
df.head(3)

Raw dataset shape: (568454, 10)
Total reviews: 568,454


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...


In [3]:
print('Starting point summary:')
print(f'  Total rows:         {len(df):,}')
print(f'  Missing Text:       {df["Text"].isnull().sum():,}')
print(f'  Missing Summary:    {df["Summary"].isnull().sum():,}')
print(f'  Missing ProfileName:{df["ProfileName"].isnull().sum():,}')
print(f'  Score range:        {df["Score"].min()} to {df["Score"].max()}')

Starting point summary:
  Total rows:         568,454
  Missing Text:       0
  Missing Summary:    27
  Missing ProfileName:26
  Score range:        1 to 5


In [4]:
# Remove duplicates
before = len(df)
df = df.drop_duplicates(subset=['UserId', 'ProductId', 'Text'])
after = len(df)

print(f'Rows before: {before:,}')
print(f'Rows after:  {after:,}')
print(f'Removed:     {before - after:,} duplicate reviews')

Rows before: 568,454
Rows after:  567,145
Removed:     1,309 duplicate reviews


In [5]:
# Handle missing values
print('Missing values after duplicate removal:')
print(df.isnull().sum()[df.isnull().sum() > 0])

Missing values after duplicate removal:
ProfileName    26
Summary        27
dtype: int64


In [6]:
# Drop rows with no review text - these are unrecoverable
before = len(df)
df = df.dropna(subset=['Text'])
after = len(df)
print(f'Dropped {before - after:,} rows with missing Text.')

# Fill missing Summary with empty string
df['Summary'] = df['Summary'].fillna('')

# Fill missing ProfileName with placeholder
df['ProfileName'] = df['ProfileName'].fillna('Anonymous')

print('\nMissing values remaining:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('No missing values.' if df.isnull().sum().sum() == 0 else '')

Dropped 0 rows with missing Text.

Missing values remaining:
Series([], dtype: int64)
No missing values.


In [7]:
before = len(df)
df = df[df['HelpfulnessNumerator'] <= df['HelpfulnessDenominator']]
after = len(df)

print(f'Removed {before - after:,} rows with invalid helpfulness data.')
print(f'Remaining rows: {after:,}')

Removed 2 rows with invalid helpfulness data.
Remaining rows: 567,143


In [8]:
df['word_count_raw'] = df['Text'].apply(lambda x: len(str(x).split()))

before = len(df)
df = df[df['word_count_raw'] >= 5]
after = len(df)

print(f'Removed {before - after:,} reviews with fewer than 5 words.')
print(f'Remaining rows: {after:,}')

Removed 3 reviews with fewer than 5 words.
Remaining rows: 567,140


Raw review text contains HTML tags, special characters, numbers, and extra whitespace. All of these add noise without adding meaning. We clean the text in a pipeline of steps, each targeting a specific type of noise.

In [9]:
# Text cleaning
def clean_text(text):
    # Remove HTML tags (common in Amazon review data)
    text = re.sub(r'<.*?>', ' ', str(text))
    
    # Decode common HTML entities
    text = text.replace('&amp;', 'and').replace('&lt;', '<').replace('&gt;', '>')
    
    # Lowercase
    text = text.lower()
    
    # Expand common contractions before removing apostrophes
    contractions = {
        "won't": "will not", "can't": "cannot", "don't": "do not",
        "doesn't": "does not", "didn't": "did not", "isn't": "is not",
        "aren't": "are not", "wasn't": "was not", "weren't": "were not",
        "hasn't": "has not", "haven't": "have not", "hadn't": "had not",
        "wouldn't": "would not", "couldn't": "could not", "shouldn't": "should not",
        "i'm": "i am", "i've": "i have", "i'll": "i will", "i'd": "i would",
        "it's": "it is", "that's": "that is", "there's": "there is",
        "they're": "they are", "they've": "they have", "they'll": "they will",
        "we're": "we are", "we've": "we have", "we'll": "we will",
        "you're": "you are", "you've": "you have", "you'll": "you will",
    }
    for contraction, expansion in contractions.items():
        text = text.replace(contraction, expansion)
    
    # Remove numbers (they rarely add semantic meaning in reviews)
    text = re.sub(r'\d+', '', text)
    
    # Remove special characters and punctuation, keep only letters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

print('clean_text function defined.')

clean_text function defined.


In [10]:
sample_texts = [
    "I <b>love</b> this product! It's great & it doesn't disappoint. 10/10 would buy again.",
    "Terrible quality... won't be buying again. I've tried 3 different batches.",
    "Can't believe how bad this is. Didn't even last 2 weeks!"
]

print('Cleaning test samples:\n')
for text in sample_texts:
    print(f'Original:  {text}')
    print(f'Cleaned:   {clean_text(text)}')
    print()

Cleaning test samples:

Original:  I <b>love</b> this product! It's great & it doesn't disappoint. 10/10 would buy again.
Cleaned:   i love this product it is great it does not disappoint would buy again

Original:  Terrible quality... won't be buying again. I've tried 3 different batches.
Cleaned:   terrible quality will not be buying again i have tried different batches

Original:  Can't believe how bad this is. Didn't even last 2 weeks!
Cleaned:   cannot believe how bad this is did not even last weeks



In [11]:
print('Cleaning review text... this may take 1-2 minutes.')
df['cleaned_text'] = df['Text'].apply(clean_text)
print('Done.')

# Quick sanity check
print(f'\nSample cleaned review:')
print(df['cleaned_text'].iloc[0])

Cleaning review text... this may take 1-2 minutes.
Done.

Sample cleaned review:
i have bought several of the vitality canned dog food products and have found them all to be of good quality the product looks more like a stew than a processed meat and it smells better my labrador is finicky and she appreciates this product better than most


In [12]:
df['word_count_clean'] = df['cleaned_text'].apply(lambda x: len(x.split()))

print('Word count comparison (raw vs cleaned):')
print(f'  Raw avg word count:     {df["word_count_raw"].mean():.1f}')
print(f'  Cleaned avg word count: {df["word_count_clean"].mean():.1f}')

# Drop any reviews that became too short after cleaning
before = len(df)
df = df[df['word_count_clean'] >= 4]
after = len(df)
print(f'\nRemoved {before - after:,} reviews that became too short after cleaning.')
print(f'Remaining: {after:,}')

Word count comparison (raw vs cleaned):
  Raw avg word count:     79.9
  Cleaned avg word count: 80.4

Removed 18 reviews that became too short after cleaning.
Remaining: 567,122


### Lemmatization
Lemmatization converts each word to its base dictionary form using vocabulary and grammar rules. Unlike stemming which crudely chops endings, lemmatization produces real words.

Examples:
- `ordered`, `ordering`, `orders` → `order`
- `running`, `ran`, `runs` → `run`  
- `better`, `best` → `good`
- `was`, `were`, `been` → `be`

This ensures our topic model and classifier treat the same concept as one token, not multiple.

In [13]:
lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words('english'))
extra_stops = {
    'product', 'one', 'get', 'also', 'would', 'like',
    'really', 'very', 'got', 'even', 'much', 'make',
    'use', 'used', 'using', 'food', 'eat',
    'will', 'not', 'do', 'did', 'have', 'has', 'had',
    'cannot', 'could', 'should', 'may', 'might',
    'amazon', 'buy', 'bought', 'order', 'ordered',
    'price', 'star', 'review', 'reviewer'
}
stop_words.update(extra_stops)

print(f'Total stopwords: {len(stop_words)}')

Total stopwords: 227


In [14]:
test = "the dogs were running faster and the ordering system ordered better products"
tokens = word_tokenize(test)
lemmatized = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 2]
print(f'Original:    {test}')
print(f'Lemmatized:  {" ".join(lemmatized)}')

Original:    the dogs were running faster and the ordering system ordered better products
Lemmatized:  dog running faster ordering system better product


In [15]:
def lemmatize_text(text):
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

print('Lemmatizing text... this will take 3-5 minutes.')
df['lemmatized_text'] = df['cleaned_text'].apply(lemmatize_text)
print('Done.')

print(f'\nSample comparison:')
print(f'Cleaned:     {df["cleaned_text"].iloc[0]}')
print(f'Lemmatized:  {df["lemmatized_text"].iloc[0]}')

Lemmatizing text... this will take 3-5 minutes.
Done.

Sample comparison:
Cleaned:     i have bought several of the vitality canned dog food products and have found them all to be of good quality the product looks more like a stew than a processed meat and it smells better my labrador is finicky and she appreciates this product better than most
Lemmatized:  several vitality canned dog product found good quality look stew processed meat smell better labrador finicky appreciates better


In [16]:
df['word_count_lemmatized'] = df['lemmatized_text'].apply(lambda x: len(x.split()))

print('Word count across pipeline stages:')
print(f'  Raw avg:         {df["word_count_raw"].mean():.1f} words')
print(f'  Cleaned avg:     {df["word_count_clean"].mean():.1f} words')
print(f'  Lemmatized avg:  {df["word_count_lemmatized"].mean():.1f} words')

# Remove any reviews that are now empty after lemmatization
before = len(df)
df = df[df['word_count_lemmatized'] >= 3]
after = len(df)
print(f'\nRemoved {before - after:,} reviews that became empty after lemmatization.')
print(f'Final remaining: {after:,}')

Word count across pipeline stages:
  Raw avg:         79.9 words
  Cleaned avg:     80.4 words
  Lemmatized avg:  35.0 words

Removed 52 reviews that became empty after lemmatization.
Final remaining: 567,070


In [17]:
# Final Dataset Preparation
# Binary sentiment label
df['sentiment'] = df['Score'].apply(lambda x: 'Positive' if x >= 4 else 'Negative')

# Helpfulness ratio (0 where no votes exist)
df['helpfulness_ratio'] = df.apply(
    lambda row: row['HelpfulnessNumerator'] / row['HelpfulnessDenominator']
    if row['HelpfulnessDenominator'] > 0 else 0.0, axis=1
)

# Convert timestamp to datetime
df['review_date'] = pd.to_datetime(df['Time'], unit='s')
df['year'] = df['review_date'].dt.year
df['month'] = df['review_date'].dt.month

print('New columns added: sentiment, helpfulness_ratio, review_date, year, month')
print(f'\nSentiment distribution:')
print(df['sentiment'].value_counts())

New columns added: sentiment, helpfulness_ratio, review_date, year, month

Sentiment distribution:
sentiment
Positive    442817
Negative    124253
Name: count, dtype: int64


In [18]:
final_cols = [
    'Id',
    'ProductId',
    'UserId',
    'Score',
    'sentiment',
    'Text',
    'cleaned_text',
    'lemmatized_text',
    'Summary',
    'helpfulness_ratio',
    'word_count_raw',
    'word_count_clean',
    'word_count_lemmatized',
    'review_date',
    'year',
    'month'
]

df_final = df[final_cols].reset_index(drop=True)

print(f'Final dataset shape: {df_final.shape}')
print(f'\nColumn list:')
for col in df_final.columns:
    print(f'  {col}')

Final dataset shape: (567070, 16)

Column list:
  Id
  ProductId
  UserId
  Score
  sentiment
  Text
  cleaned_text
  lemmatized_text
  Summary
  helpfulness_ratio
  word_count_raw
  word_count_clean
  word_count_lemmatized
  review_date
  year
  month


In [19]:
# Save and Validate
output_path = '../data/processed/reviews_clean.csv'
df_final.to_csv(output_path, index=False)
print(f'Saved to {output_path}')
print(f'File contains {len(df_final):,} rows and {df_final.shape[1]} columns.')

Saved to ../data/processed/reviews_clean.csv
File contains 567,070 rows and 16 columns.


In [20]:
df_check = pd.read_csv(output_path)
print('Validation check on saved file:')
print(f'  Shape:            {df_check.shape}')
print(f'  Missing values:   {df_check.isnull().sum().sum()}')
print(f'  Sentiment counts: {df_check["sentiment"].value_counts().to_dict()}')
print(f'\nSample cleaned review:')
print(df_check["cleaned_text"].iloc[100])
print(f'\nSample lemmatized review:')
print(df_check["lemmatized_text"].iloc[100])

Validation check on saved file:
  Shape:            (567070, 16)
  Missing values:   27
  Sentiment counts: {'Positive': 442817, 'Negative': 124253}

Sample cleaned review:
arrived slightly thawed my parents would not accept it however the company was very helpful and issued a full refund

Sample lemmatized review:
arrived slightly thawed parent accept however company helpful issued full refund


**Preprocessing Pipeline Summary:**

| Stage | Rows Remaining | Action Taken |
|---|---|---|
| Raw data | 568,454 | Loaded from Reviews.csv |
| After duplicate removal | ~568,000 | Removed same user/product/text duplicates |
| After missing text removal | ~568,000 | Dropped rows with no review text |
| After invalid helpfulness removal | ~567,000 | Removed numerator > denominator rows |
| After short review removal | 567,122 | Removed reviews under 5 words |
| After cleaning short removal | 567,122 | Removed reviews under 4 words post-clean |
| After lemmatization | 567,070 | Removed empty reviews post-lemmatization |

**Output:** `data/processed/reviews_clean.csv` with three text versions per review: raw `Text`, `cleaned_text`, and `lemmatized_text`.